# Grouped Query Attention (GQA)

**Paper:** GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints (Ainslie et al. 2023)

**Used in:** LLaMA-2 (70B), Mistral-7B, Gemma, Falcon, Phi-3 — basically every production LLM.

## The Problem: KV Cache grows with heads

During inference, we cache K and V for all previous tokens (KV cache).
With Multi-Head Attention (MHA), we cache K and V for **every head**.

```
Model: LLaMA-2 70B,  n_heads=64,  d_k=128,  seq_len=4096, batch=1
KV cache per layer = 2 × 64 × 128 × 4096 × float16 = 128 MB per layer
80 layers × 128 MB = 10 GB just for the KV cache!
```

## Three attention variants

```
MHA  — Multi-Head Attention:    Q heads = K heads = V heads = H
                                 every head has its own K and V

MQA  — Multi-Query Attention:   Q heads = H,  K heads = V heads = 1
                                 all Q heads share ONE K and V head
                                 fastest, but quality drops

GQA  — Grouped Query Attention: Q heads = H,  K/V heads = G  (where 1 < G < H)
                                 Q heads are divided into G groups
                                 each group shares one K and V head
                                 best quality/speed tradeoff ← the sweet spot
```

LLaMA-2-70B: H=64 query heads, G=8 KV heads → 8x smaller KV cache.

## Resources

| Resource | Link |
|---|---|
| Original paper — GQA (Ainslie et al. 2023) | https://arxiv.org/abs/2305.13245 |
| MQA paper (Shazeer 2019, precursor) | https://arxiv.org/abs/1911.02150 |
| LLaMA-2 paper (GQA in production) | https://arxiv.org/abs/2307.09288 |
| Mistral-7B paper (GQA + sliding window) | https://arxiv.org/abs/2310.06825 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        """
        d_model:    total model dimension
        n_heads:    number of query heads  (e.g., 32)
        n_kv_heads: number of KV heads     (e.g., 8  for GQA, 1 for MQA, 32 for MHA)
        """
        super().__init__()
        assert n_heads % n_kv_heads == 0, 'n_heads must be divisible by n_kv_heads'

        self.n_heads      = n_heads
        self.n_kv_heads   = n_kv_heads
        self.n_rep        = n_heads // n_kv_heads   # how many Q heads share each KV head
        self.d_k          = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads    * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)  # fewer K heads
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)  # fewer V heads
        self.W_o = nn.Linear(n_heads * self.d_k, d_model,    bias=False)

    def forward(self, x):
        batch, seq_len, _ = x.shape

        # Project to Q, K, V
        Q = self.W_q(x).reshape(batch, seq_len, self.n_heads,    self.d_k).transpose(1, 2)
        K = self.W_k(x).reshape(batch, seq_len, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).reshape(batch, seq_len, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Expand K and V to match Q heads by repeating each KV head n_rep times
        # [batch, n_kv_heads, seq, d_k] → [batch, n_heads, seq, d_k]
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)

        # Standard attention from here — same as MHA
        scores  = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        out     = (weights @ V).transpose(1, 2).reshape(batch, seq_len, -1)
        return self.W_o(out)


batch, seq_len, d_model = 2, 64, 512

mha  = GroupedQueryAttention(d_model, n_heads=8, n_kv_heads=8)  # standard MHA
gqa  = GroupedQueryAttention(d_model, n_heads=8, n_kv_heads=2)  # GQA: 4 Q heads per KV
mqa  = GroupedQueryAttention(d_model, n_heads=8, n_kv_heads=1)  # MQA: all share 1 KV

x = torch.randn(batch, seq_len, d_model)

for name, model in [('MHA', mha), ('GQA', gqa), ('MQA', mqa)]:
    out = model(x)
    kv_params = sum(p.numel() for name_, p in model.named_parameters() if 'k' in name_ or 'v' in name_)
    print(f'{name}: output {out.shape} | KV params: {kv_params:,}')

In [ ]:
# KV Cache size comparison at inference time
print('KV Cache size comparison (LLaMA-2 style, float16, seq_len=4096):\n')

for name, n_kv_heads in [('MHA (n_kv=32)', 32), ('GQA (n_kv=8)', 8), ('MQA (n_kv=1)', 1)]:
    n_layers  = 32
    d_model   = 4096
    n_heads   = 32
    d_k       = d_model // n_heads
    seq_len   = 4096
    bytes_per = 2  # float16

    # K cache + V cache, per layer
    kv_cache_per_layer = 2 * n_kv_heads * d_k * seq_len * bytes_per
    kv_cache_total_gb  = kv_cache_per_layer * n_layers / 1e9

    print(f'{name:20s}: {kv_cache_total_gb:.2f} GB')